In [ ]:
import numpy as np
import pandas as pd

from sample_db import SampleDB

db = SampleDB()

In [ ]:
[str(v) for v in range(200, 210)]

In [ ]:
cv_values = []

for run in [str(v) for v in range(200, 210)]:
    cv_values.extend(db.cv_values_for(run))
cv_values = np.vstack(cv_values)
cv_values = cv_values[:, :3]

cv_values = pd.DataFrame(cv_values, columns=['a_cv', 'b_cv', 'morph'])
cv_values.head()

In [ ]:
import plotly.graph_objects as go

points = cv_values[["a_cv", "b_cv", "morph"]].to_numpy()
bins = 24

hist, edges = np.histogramdd(points, bins=bins)
occupied = np.argwhere(hist > 0)
counts = hist[hist > 0]

centers = [0.5 * (axis_edges[1:] + axis_edges[:-1]) for axis_edges in edges]
xyz = np.column_stack(
    [centers[dim][occupied[:, dim]] for dim in range(3)]
)

count_min = counts.min()
count_max = counts.max()
denom = (count_max - count_min) if count_max > count_min else 1.0
sizes = 4.0 + 8.0 * (counts - count_min) / denom

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=xyz[:, 0],
            y=xyz[:, 1],
            z=xyz[:, 2],
            mode="markers",
            marker=dict(
                color=counts,
                colorscale="Viridis",
                colorbar=dict(title="Points per bin"),
                opacity=0.75,
            ),
            text=[f"count={int(c)}" for c in counts],
            hovertemplate="a_cv=%{x:.3f}<br>b_cv=%{y:.3f}<br>morph=%{z:.3f}<br>%{text}<extra></extra>",
        )
    ]
)

fig.update_layout(
    title=f"3D CV Density (occupied bins: {len(counts)}, bins per axis: {bins})",
    scene=dict(
        xaxis_title="a_cv",
        yaxis_title="b_cv",
        zaxis_title="morph",
    ),
    margin=dict(l=0, r=0, b=0, t=45),
)

fig.show()